# Attention 算子库 — 一键跑分

手写 Triton attention kernel 全场景闭环：prefill → causal → decode → GQA → PagedAttention → backward。

**先选 GPU**：`运行时 → 更改运行时类型 → T4 / L4 / A100 GPU`。

`bench.py` 在**运行时实测**当前 GPU 的带宽和 fp16 峰值再算 `% 峰值`，分母口径统一。
注意：免费 Colab 的 T4(sm_75) 上 Triton 3.8 的 `tl.dot` 后端退化，功能全对但 `% 峰值` 只有个位数（见 README「T4 上的性能异常」）；**看性能请用本地 Ampere 的数字**。

按顺序 Run All 即可。第一步自动从 GitHub clone 本仓库；网络不通时可改用上传 zip / Drive 兜底。

In [ ]:
# Cell 1: 环境检查
import torch, sys
print('Python', sys.version.split()[0], '| torch', torch.__version__)
if not torch.cuda.is_available():
    raise SystemExit('❌ 没检测到 GPU。请到 运行时 → 更改运行时类型 → 选 T4/A100/L4，然后重连。')
p = torch.cuda.get_device_properties(0)
print(f'✅ GPU: {p.name} | SM {p.multi_processor_count} | 显存 {p.total_memory/1024**3:.0f} GB')
try:
    import triton
    print('triton', triton.__version__)
except ImportError:
    print('triton 未装，下一步会装')

In [ ]:
# Cell 2: 获取代码（默认 GitHub clone，带重试；网络不通可改 zip / Drive）
import os, shutil, subprocess, time

REPO = 'https://github.com/AwakeZz/attention-lab.git'
DIR  = '/content/attention-lab'  # 绝对路径: Cell 4 会 chdir, 相对路径会嵌套出错

def _complete(d):
    """代码是否完整可用: 有 bench.py 且 kernels/ 下 >=7 个 kernel。"""
    kd = os.path.join(d, 'kernels')
    return (os.path.isfile(os.path.join(d, 'bench.py')) and os.path.isdir(kd)
            and len([f for f in os.listdir(kd) if f.endswith('.py')]) >= 7)

def from_github():
    # 关键: 已完整就直接复用, 绝不先删。旧版是先 rmtree 再 clone,
    # Colab->GitHub 偶尔抖动 (exit 128), 一失败就把好代码也删了。
    if _complete(DIR):
        print('✅ 本地代码已完整, 直接复用 (想强制更新就手动删掉 attention-lab 目录再跑)')
        return
    tmp = DIR + '_new'
    if os.path.isdir(tmp):
        shutil.rmtree(tmp)
    last = ''
    for i in range(1, 6):
        try:
            subprocess.run(['git', 'clone', '--depth', '1', REPO, tmp],
                           check=True, capture_output=True, text=True, timeout=180)
            if not _complete(tmp):
                raise RuntimeError('clone 成功但代码不完整')
            if os.path.isdir(DIR):
                shutil.rmtree(DIR)
            os.rename(tmp, DIR)
            print(f'✅ 代码来自 GitHub (第 {i} 次尝试成功)')
            return
        except Exception as e:
            last = str(e)[:160]
            print(f'  第 {i}/5 次 clone 失败: {last}')
            time.sleep(3)
    if _complete(DIR):
        print('⚠️ clone 一直失败, 沿用本地已有代码 (可能不是最新版)')
    else:
        raise RuntimeError(f'取代码失败, 请改用 zip/Drive 兜底。最后错误: {last}')

def from_zip():
    # 左侧文件栏上传 attention-lab.zip（本项目目录压缩），再运行本函数
    import zipfile, glob
    z = glob.glob('attention-lab*.zip')[0]
    with zipfile.ZipFile(z) as f:
        f.extractall('.')
    print('✅ 代码来自上传的 zip:', z)

def from_drive():
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree('/content/drive/MyDrive/attention-lab', DIR, dirs_exist_ok=True)
    print('✅ 代码来自 Google Drive')

MODE = 'github'   # 可选 'github' / 'zip' / 'drive'
try:
    {'github': from_github, 'zip': from_zip, 'drive': from_drive}[MODE]()
except Exception as e:
    print(f'⚠️ {MODE} 取代码失败: {e}')
    print('兜底：左侧上传 attention-lab.zip，把下方 MODE 改成 zip 后重跑本格。')
    raise


In [ ]:
# Cell 3: 检查 triton（kernel 只用 3.x 稳定 API；缺关键符号才升级）
import triton, subprocess, sys
if not hasattr(triton, 'next_power_of_2'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'triton'], check=True)
    print('已升级 triton，建议 运行时 → 重启运行时 后从 Cell 1 重跑')
else:
    print('✅ triton 版本兼容:', triton.__version__)

In [ ]:
# Cell 4: 跑全部 benchmark（实测硬件峰值 + 正确性 + 对标 SDPA）
import os
os.chdir('/content/attention-lab')  # 绝对路径, 不依赖上次运行留下的 cwd
!python bench.py

---

### 结果怎么看

- **% 峰值**：除以本卡实测的 fp16 GEMM 峰值。prefill 类（13–15、18）compute-bound，这个数有意义；decode 类（16/17/19）memory-bound，只有 10–47% 是正常的，看延迟才对。
- **SDPA 加速比**：第 17 题那个 40×+ 不要当成绩讲——PyTorch 在 HQ≠HKV 时掉 math 后端。真材料和每个手段的单独收益见 `README.md`。

想只跑某几题：`!python bench.py --only 14 16`。

⚠️ 用完记得 `运行时 → 断开连接` 释放 GPU——计算单元对空闲的 GPU 运行时也照扣。